# 08 · Zero-shot detectors: Binoculars, Fast-DetectGPT, log-likelihood, log-rank
**Purpose (RQ4, RQ5):** score every evaluation text with two **scorer pairs** that share a tokenizer:
- `generic`: Qwen3.5-4B-Base (observer) / Qwen3.5-4B (performer)
- `kazakh`: ISSAI Qwen3.5-4B-Base-Kazakh / Qwen3.5-4B-Kazakh (Kazakh-adapted continuation of the same base family)

Comparing the two isolates the effect of **language adaptation of the scoring LM** (RQ5).

| Score | Definition (higher ⇒ more likely machine) | Source |
|---|---|---|
| Binoculars | −(log-PPL_performer / cross-PPL(observer→performer)) — as in the official implementation | Hans et al., ICML 2024 |
| Fast-DetectGPT | analytic sampling discrepancy, observer as reference and scorer | Bao et al., ICLR 2024 |
| log-likelihood | mean token log-prob (observer) | — |
| log-rank | −mean log-rank (observer) | — |

**Framework note:** inference-only scoring uses Hugging Face `transformers` (documented exception: no training happens here; all trained detectors use Keras/KerasHub).
**Runtime:** A100 (40 GB) recommended — two ~4–5B models in bf16 plus full-vocabulary logits; on L4 set `batch_size` to 2. ~30–60 min per pair. Resumable (partial scores cached).

In [ ]:
# ==== Bootstrap (identical in every notebook) ====
NOTEBOOK = "08_zero_shot_detectors"
PROJECT_ROOT = "/content/drive/MyDrive/multilingual-ai-text-detection"   # <- edit if needed
import os, json
try:
    from google.colab import drive; drive.mount("/content/drive")
except ImportError:
    pass
os.environ["MATD_ROOT"] = PROJECT_ROOT
def load_common(path=f"{PROJECT_ROOT}/notebooks/00_common.ipynb"):
    nb = json.load(open(path, encoding="utf-8"))
    code = "\n\n".join("".join(c["source"]) for c in nb["cells"] if c["cell_type"] == "code")
    exec(compile(code, path, "exec"), globals())
load_common()
log = get_logger(NOTEBOOK); env = env_snapshot(NOTEBOOK)

In [ ]:
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
ZS = CFG["zeroshot"]; dev = "cuda"
assert torch.cuda.is_available(), "GPU required"
DTYPE = torch.bfloat16 if ZS["dtype"] == "bfloat16" else torch.float16
EVAL = load_eval_texts()
log.info(f"texts to score: {len(EVAL)}")

def load_lm(name):
    """Causal LM; falls back to the image-text-to-text class for multimodal checkpoints (text-only forward)."""
    try:
        return AutoModelForCausalLM.from_pretrained(name, torch_dtype=DTYPE, device_map=dev).eval()
    except (ValueError, KeyError) as e:
        log.warning(f"{name}: AutoModelForCausalLM failed ({e}); trying AutoModelForImageTextToText")
        from transformers import AutoModelForImageTextToText
        return AutoModelForImageTextToText.from_pretrained(name, torch_dtype=DTYPE, device_map=dev).eval()

@torch.no_grad()
def score_batch(texts, tok, obs, perf):
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=ZS["max_tokens"]).to(dev)
    lo = obs(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits[:, :-1].float()
    lp = perf(input_ids=enc.input_ids, attention_mask=enc.attention_mask).logits[:, :-1].float()
    V = min(lo.shape[-1], lp.shape[-1]); lo, lp = lo[..., :V], lp[..., :V]   # guard against padded vocab rows
    tgt = enc.input_ids[:, 1:]; mask = enc.attention_mask[:, 1:].float(); n = mask.sum(1).clamp(min=1)
    lso, lsp = F.log_softmax(lo, -1), F.log_softmax(lp, -1); po = lso.exp()
    ll_o = lso.gather(-1, tgt[..., None]).squeeze(-1)
    ll_p = lsp.gather(-1, tgt[..., None]).squeeze(-1)
    # Binoculars (official code: ppl from performer logits; x-ppl = CE(softmax(observer), log_softmax(performer)))
    ppl = -(ll_p * mask).sum(1) / n
    xppl = -((po * lsp).sum(-1) * mask).sum(1) / n
    bino = ppl / xppl
    # Fast-DetectGPT analytic criterion (reference = scorer = observer)
    mu = (po * lso).sum(-1); var = (po * lso.pow(2)).sum(-1) - mu.pow(2)
    fdg = ((ll_o - mu) * mask).sum(1) / (var * mask).sum(1).clamp(min=1e-6).sqrt()
    # log-likelihood and log-rank (observer)
    rank = (lo > lo.gather(-1, tgt[..., None])).sum(-1).float() + 1
    loglik = (ll_o * mask).sum(1) / n
    logrank = (rank.log() * mask).sum(1) / n
    return {"binoculars": (-bino).cpu().numpy(), "fastdetect": fdg.cpu().numpy(),
            "loglik": loglik.cpu().numpy(), "logrank": (-logrank).cpu().numpy(), "n_tokens": n.cpu().numpy()}

In [ ]:
METHODS = ["binoculars", "fastdetect", "loglik", "logrank"]
for pair, names in ZS["pairs"].items():
    rids = {m: make_run_id("E5", f"{m}-{pair}", "none", "none", 0, extra=names) for m in METHODS}
    if all(is_done(r) for r in rids.values()):
        log.info(f"pair {pair}: done"); continue
    for r in rids.values():
        register_run(r, "running", exp="E5", detector=r.split("__")[1], train="none", heldout="none", seed=0,
                     params_json=json.dumps({"pair": pair, **names, "max_tokens": ZS["max_tokens"]}))
    t0 = time.time()
    tok = AutoTokenizer.from_pretrained(names["observer"]); tok.padding_side = "right"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    assert tok.get_vocab() == AutoTokenizer.from_pretrained(names["performer"]).get_vocab(), "pair must share a tokenizer"
    obs, perf = load_lm(names["observer"]), load_lm(names["performer"])
    part = P["predictions"] / f"_partial_zeroshot_{pair}.parquet"
    have = pd.read_parquet(part) if part.exists() else pd.DataFrame()
    todo = EVAL[~EVAL.doc_id.isin(set(have.doc_id))] if len(have) else EVAL
    # sort by length for efficient padding
    todo = todo.assign(_len=todo.text.str.len()).sort_values("_len")
    chunks, bs = [], ZS["batch_size"]
    for i in range(0, len(todo), bs):
        b = todo.iloc[i:i + bs]
        s = score_batch(b.text.tolist(), tok, obs, perf)
        chunks.append(pd.DataFrame({"doc_id": b.doc_id.values, **s}))
        if len(chunks) % 200 == 0 or i + bs >= len(todo):
            have = pd.concat([have] + chunks, ignore_index=True); chunks = []
            save_parquet(have, part); log.info(f"[{pair}] scored {len(have)}/{len(EVAL)}")
    for m, r in rids.items():
        save_parquet(have[["doc_id", m]].rename(columns={m: "score"}).assign(prob=np.nan), pred_path(r))
        register_run(r, "done", seconds=round(time.time() - t0, 1), notes=json.dumps({"n_scored": len(have)}))
    del obs, perf; gc.collect(); torch.cuda.empty_cache()

Quick sanity check: AUROC on the clean test split per language (full analysis in 09/10).

In [ ]:
reg = load_registry(); zs = reg[(reg.exp == "E5") & (reg.status == "done")]
test = EVAL[EVAL.eval_set == "test"][["doc_id", "lang", "label"]]
out = []
for r in zs.itertuples():
    s = pd.read_parquet(pred_path(r.run_id)).merge(test, on="doc_id")
    out.append({"detector": r.detector, **{l: round(auroc(g.label, g.score), 3) for l, g in s.groupby("lang")}})
pd.DataFrame(out)